# Whisper-Small Hindi Fine-Tuning Pipeline

**End-to-end pipeline** for fine-tuning OpenAI's `whisper-small` on Hindi speech data.

| Step | Description |
|------|-------------|
| 1 | Setup and Installation |
| 2 | Configuration |
| 3 | Load Dataset from Excel |
| 4 | Audio Preprocessing |
| 5 | Text Normalization |
| 6 | Data Quality Checks |
| 7 | Dataset Formatting for Whisper |
| 8 | Baseline Evaluation (FLEURS Hindi) |
| 9 | Fine-Tuning |
| 10 | Post-Training Evaluation and WER Report |
| 11 | Save / Push Model |

> **Runtime**: Go to *Runtime > Change runtime type > GPU (T4)* before running.

---
## 1 - Setup and Installation

In [ ]:
!pip install -q \
    transformers>=4.37.0 \
    datasets>=2.16.0 \
    accelerate>=0.26.0 \
    evaluate>=0.4.1 \
    jiwer>=3.0.3 \
    soundfile \
    librosa \
    torchaudio \
    huggingface_hub \
    pandas \
    openpyxl

print("All packages installed.")

In [ ]:
import os, json, re, unicodedata, warnings, shutil
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torchaudio
import librosa
import soundfile as sf
import evaluate
from tqdm.auto import tqdm

from datasets import Dataset, DatasetDict, Audio, load_dataset
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    WhisperTokenizer,
    WhisperFeatureExtractor,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    EarlyStoppingCallback,
)
from dataclasses import dataclass
from typing import Any, Dict, List, Union

warnings.filterwarnings("ignore")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}  |  Torch: {torch.__version__}")

---
## 2 - Configuration

Edit the values below to match **your** dataset.

In [ ]:
MODEL_NAME       = "openai/whisper-small"
LANGUAGE         = "hi"           
TASK             = "transcribe"


EXCEL_FILE       = "dataset.xlsx"   


AUDIO_COL        = "audio_path"     
TEXT_COL         = "transcription"   

PROCESSED_DIR    = Path("./processed")      
OUTPUT_DIR       = Path("./whisper-hi-finetuned")

LEARNING_RATE    = 1e-5
TRAIN_BATCH      = 16
EVAL_BATCH       = 8
NUM_EPOCHS       = 10
WARMUP_STEPS     = 500
GRAD_ACCUM       = 2
MAX_GRAD_NORM    = 1.0
FP16             = True  
TRAIN_VAL_SPLIT  = 0.1   


TARGET_SR        = 16_000
MIN_DURATION_SEC = 1.0
MAX_DURATION_SEC = 30.0   

for d in [PROCESSED_DIR, OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Configuration set.")

---
## 3 - Load Dataset from Excel

Reads your **single Excel file** containing audio paths and Hindi transcriptions.

> **Steps:**
> 1. Upload your `.xlsx` file using the Colab file browser (folder icon on left)
> 2. Make sure `EXCEL_FILE`, `AUDIO_COL`, and `TEXT_COL` in Section 2 match your file
> 3. Run this cell

In [ ]:

df = pd.read_excel(EXCEL_FILE)

print(f"Excel loaded: {len(df)} rows")
print(f"Columns found: {df.columns.tolist()}")
print()
print(df.head())

---
## 3.1 - Fix Faulty Dataset URLs

The original dataset URLs use an **outdated base path** (`joshtalks-data-collection/hq_data/hi/`).
This cell automatically converts **all URLs** in the DataFrame to the correct path (`upload_goai/`).

| Type | URL Pattern |
|------|------------|
| **Old (broken)** | `storage.googleapis.com/joshtalks-data-collection/hq_data/hi/{id}/{file}` |
| **New (correct)** | `storage.googleapis.com/upload_goai/{id}/{file}` |


In [ ]:


import re

# Base path mapping
OLD_BASE = "https://storage.googleapis.com/joshtalks-data-collection/hq_data/hi/"
NEW_BASE = "https://storage.googleapis.com/upload_goai/"


def fix_url(url_string):
    """
    Convert a single URL from the old (faulty) format to the new (correct) format.

    Example:
      OLD: https://storage.googleapis.com/joshtalks-data-collection/hq_data/hi/967179/825780_transcription.json
      NEW: https://storage.googleapis.com/upload_goai/967179/825780_transcription.json
    """
    if not isinstance(url_string, str):
        return url_string
    return url_string.replace(OLD_BASE, NEW_BASE)


def fix_all_urls_in_dataframe(df):
    """
    Scan every column in the DataFrame for URLs matching the old base path
    and replace them with the corrected base path.
    Handles: transcription URLs, audio URLs, metadata URLs, etc.
    """
    total_fixed = 0

    for col in df.columns:
        
        if df[col].dtype == 'object':
            mask = df[col].astype(str).str.contains(OLD_BASE, na=False)
            count = mask.sum()
            if count > 0:
                df[col] = df[col].apply(fix_url)
                total_fixed += count
                print(f"  [✓] Column '{col}': fixed {count} URLs")

    return df, total_fixed


print("Fixing faulty URLs in dataset...")
print(f"  Old base: {OLD_BASE}")
print(f"  New base: {NEW_BASE}")
print()

df, num_fixed = fix_all_urls_in_dataframe(df)

if num_fixed > 0:
    print(f"\n[✓] Done! Fixed {num_fixed} URLs across all columns.")
else:
    print("\n[INFO] No faulty URLs found — dataset URLs are already correct.")

print()
print("Sample URLs after fix:")
for col in df.columns:
    if df[col].dtype == 'object':
        sample = df[col].dropna().head(1)
        if not sample.empty and str(sample.iloc[0]).startswith('http'):
            print(f"  {col}: {sample.iloc[0][:100]}")


In [ ]:
def find_column(df, candidates, label):
    """Try to find a column matching one of the candidate names (case-insensitive)."""
    df_cols_lower = {c.strip().lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in df_cols_lower:
            return df_cols_lower[cand.lower()]
    return None

audio_candidates = [
    AUDIO_COL, "audio_path", "audio_url", "audio", "file", "filepath",
    "file_path", "filename", "path", "url", "audio_file", "wav", "wav_path",
    "recording", "recording_path", "sound", "sound_path",
]

text_candidates = [
    TEXT_COL, "transcription", "transcript", "text", "sentence",
    "hindi_text", "label", "ground_truth", "reference",
    "caption", "description", "utterance",
]

detected_audio = find_column(df, audio_candidates, "audio")
detected_text  = find_column(df, text_candidates, "text")

if detected_audio:
    AUDIO_COL = detected_audio
    print(f"Audio column detected: '{AUDIO_COL}'")
else:
    print(f"WARNING: Could not auto-detect audio column. Set AUDIO_COL manually.")
    print(f"  Available columns: {df.columns.tolist()}")

if detected_text:
    TEXT_COL = detected_text
    print(f"Text column detected:  '{TEXT_COL}'")
else:
    print(f"WARNING: Could not auto-detect text column. Set TEXT_COL manually.")
    print(f"  Available columns: {df.columns.tolist()}")

In [ ]:
def build_manifest_from_excel(df, audio_col, text_col):
    """
    Convert Excel DataFrame into a manifest dict.
    Returns { index: {"audio_path": str, "transcript": str, "metadata": {}} }
    """
    manifest = {}
    skipped = 0

    for idx, row in df.iterrows():
        audio_path = str(row[audio_col]).strip()
        transcript = str(row[text_col]).strip()

        if not audio_path or audio_path == "nan" or not transcript or transcript == "nan":
            skipped += 1
            continue

        meta = {}
        for col in df.columns:
            if col not in [audio_col, text_col]:
                val = row[col]
                if pd.notna(val):
                    meta[col] = val

        manifest[str(idx)] = {
            "audio_path": audio_path,
            "transcript": transcript,
            "metadata": meta,
        }

    print(f"Manifest built: {len(manifest)} valid samples ({skipped} skipped)")
    return manifest


manifest = build_manifest_from_excel(df, AUDIO_COL, TEXT_COL)

for k in list(manifest.keys())[:3]:
    print(f"  [{k}] audio: {manifest[k]['audio_path'][:80]}")
    print(f"        text:  {manifest[k]['transcript'][:80]}")
    print()

---
## 4 - Audio Preprocessing

- Convert to **16 kHz, mono, WAV**
- Filter clips shorter than 1 s or longer than 30 s
- Verify duration consistency between metadata and actual audio

In [ ]:
def preprocess_audio(manifest: Dict[str, dict]) -> Dict[str, dict]:
    """
    Resample -> mono -> WAV.  Returns cleaned manifest.
    """
    cleaned = {}
    stats = Counter()

    for rid, entry in tqdm(manifest.items(), desc="Audio preprocessing"):
        src = entry["audio_path"]
        dst = PROCESSED_DIR / f"{rid}.wav"

        try:
            waveform, sr = torchaudio.load(src)
        except Exception:
            stats["corrupted"] += 1
            continue

        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)

        if sr != TARGET_SR:
            resampler = torchaudio.transforms.Resample(sr, TARGET_SR)
            waveform = resampler(waveform)

        duration = waveform.shape[1] / TARGET_SR

        if duration < MIN_DURATION_SEC:
            stats["too_short"] += 1
            continue
        if duration > MAX_DURATION_SEC:
            stats["too_long"] += 1
            continue

        meta_dur = entry.get("metadata", {}).get("duration", None)
        if meta_dur is not None and abs(float(meta_dur) - duration) > 1.0:
            stats["duration_mismatch"] += 1
            continue

        torchaudio.save(str(dst), waveform, TARGET_SR)

        cleaned[rid] = {
            **entry,
            "audio_path": str(dst),
            "duration": round(duration, 2),
        }
        stats["ok"] += 1

    print(f"\nAudio preprocessing stats: {dict(stats)}")
    print(f"{len(cleaned)} samples ready.")
    return cleaned

manifest = preprocess_audio(manifest)

---
## 5 - Text Normalization

- Unicode NFC normalization
- Remove extra whitespace and non-speech artifacts
- Standardize punctuation
- Preserve **Devanagari script** strictly
- Lowercasing for Latin characters only

In [ ]:

_DEVANAGARI_RE = re.compile(r"[\u0900-\u097F]")
_NON_SPEECH    = re.compile(r"\[.*?\]|\(.*?\)|\{.*?\}")  
_MULTI_SPACE   = re.compile(r"\s+")

_PUNCT_MAP = str.maketrans({
    "\u0964": "\u0964",   
    "\u0965": "\u0965",   
    "\u201c": '"',         
    "\u201d": '"',         
    "\u2018": "'",         
    "\u2019": "'",        
    "\u2026": "...",       
    "\u2013": "-",         
    "\u2014": "-",         
})


def normalize_text(text: str) -> str:
    """Clean and normalize a Hindi transcription string."""
    
    text = unicodedata.normalize("NFC", text)

    text = _NON_SPEECH.sub("", text)

    text = text.translate(_PUNCT_MAP)

    result = []
    for ch in text:
        if _DEVANAGARI_RE.match(ch) or not ch.isalpha():
            result.append(ch)
        else:
            result.append(ch.lower())
    text = "".join(result)

    text = _MULTI_SPACE.sub(" ", text).strip()

    return text


# Quick demo
_demo = "  \u092f\u0939   \u090f\u0915 [laughter] \u091f\u0947\u0938\u094d\u091f  \u0939\u0948\u0964  "
print(f"Before: '{_demo}'")
print(f"After : '{normalize_text(_demo)}'")

In [ ]:
def apply_text_normalization(manifest: Dict[str, dict]) -> Dict[str, dict]:
    """Apply normalize_text to every transcript in the manifest."""
    for rid in manifest:
        manifest[rid]["transcript"] = normalize_text(manifest[rid]["transcript"])
    print(f"Text normalized for {len(manifest)} samples.")
    return manifest

manifest = apply_text_normalization(manifest)

---
## 6 - Data Quality Checks

- Remove missing / empty transcription
- Remove corrupted audio
- Verify `language == "hi"`

In [ ]:
def quality_filter(manifest: Dict[str, dict]) -> Dict[str, dict]:
    """Drop samples that fail quality checks."""
    cleaned = {}
    reasons = Counter()

    for rid, entry in manifest.items():
        
        if not entry["transcript"].strip():
            reasons["empty_transcript"] += 1
            continue

        ap = Path(entry["audio_path"])
        if not ap.exists():
            reasons["missing_audio"] += 1
            continue
        try:
            info = torchaudio.info(str(ap))
        except Exception:
            reasons["corrupted_audio"] += 1
            continue
        
        lang = entry.get("metadata", {}).get("language", "hi")
        if lang != "hi":
            reasons["wrong_language"] += 1
            continue

        cleaned[rid] = entry

    print(f"Quality filter:  kept {len(cleaned)} / {len(manifest)}")
    if reasons:
        print(f"   Dropped: {dict(reasons)}")
    return cleaned

manifest = quality_filter(manifest)
print(f"\nFinal dataset size: {len(manifest)} samples")

---
## 7 - Dataset Formatting for Whisper

- Build HuggingFace `Dataset`
- Generate `input_features` (log-mel spectrogram) and `labels` (tokenized text)
- 90 / 10 train-validation split

In [ ]:

processor = WhisperProcessor.from_pretrained(
    MODEL_NAME, language=LANGUAGE, task=TASK
)
feature_extractor = processor.feature_extractor
tokenizer         = processor.tokenizer

print(f"Processor loaded for {MODEL_NAME} | lang={LANGUAGE}")

In [ ]:
def manifest_to_hf_dataset(manifest: Dict[str, dict]) -> Dataset:
    """Convert manifest dict -> HuggingFace Dataset with audio column."""
    records = [
        {"audio": entry["audio_path"], "sentence": entry["transcript"]}
        for entry in manifest.values()
    ]
    ds = Dataset.from_list(records)
    ds = ds.cast_column("audio", Audio(sampling_rate=TARGET_SR))
    return ds


def prepare_dataset(batch):
    """Map function: audio -> input_features, text -> labels."""
    audio = batch["audio"]

    batch["input_features"] = feature_extractor(
        audio["array"], sampling_rate=audio["sampling_rate"]
    ).input_features[0]

    batch["labels"] = tokenizer(batch["sentence"]).input_ids

    return batch

In [ ]:

ds_full = manifest_to_hf_dataset(manifest)
ds_full = ds_full.map(prepare_dataset, remove_columns=ds_full.column_names, num_proc=1)

split = ds_full.train_test_split(test_size=TRAIN_VAL_SPLIT, seed=42)
dataset = DatasetDict({
    "train": split["train"],
    "validation": split["test"],
})

print(f"Dataset ready -- Train: {len(dataset['train'])}  |  Val: {len(dataset['validation'])}")

---
## 8 - Baseline Evaluation on Your Dataset

Evaluate the **pretrained** Whisper-small on your own dataset's validation split
to get a baseline WER *before* fine-tuning.

> Uses the same 90/10 train-validation split created in Section 7.


In [ ]:

eval_dataset = dataset["validation"]
print(f"Evaluation dataset: {len(eval_dataset)} samples (from your Excel data)")


In [ ]:
wer_metric = evaluate.load("wer")


def evaluate_model(model, processor, eval_ds, batch_size=8):
    """
    Run inference on the evaluation dataset and return WER.
    Works with the validation split from your Excel dataset.
    """
    model.eval()
    all_preds, all_refs = [], []

    for i in tqdm(range(0, len(eval_ds), batch_size), desc="Evaluating"):
        batch = eval_ds[i : i + batch_size]

        inputs = processor.feature_extractor(
            [a["array"] for a in batch["audio"]],
            sampling_rate=TARGET_SR,
            return_tensors="pt",
            padding=True,
        )
        input_features = inputs.input_features.to(device)

        with torch.no_grad():
            predicted_ids = model.generate(
                input_features,
                language=LANGUAGE,
                task=TASK,
            )

        preds = processor.tokenizer.batch_decode(predicted_ids, skip_special_tokens=True)
        refs  = batch["sentence"]

        preds = [normalize_text(p) for p in preds]
        refs  = [normalize_text(r) for r in refs]

        all_preds.extend(preds)
        all_refs.extend(refs)

    wer = wer_metric.compute(predictions=all_preds, references=all_refs)
    return round(wer * 100, 2)


In [ ]:

model_pretrained = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME).to(device)
model_pretrained.config.forced_decoder_ids = processor.get_decoder_prompt_ids(
    language=LANGUAGE, task=TASK
)

baseline_wer = evaluate_model(model_pretrained, processor, eval_dataset)
print(f"\nBaseline WER (pretrained Whisper-small on your dataset): {baseline_wer}%")

del model_pretrained
torch.cuda.empty_cache()


---
## 9 - Fine-Tuning

| Param | Value |
|-------|-------|
| Optimizer | AdamW |
| LR | 1e-5 |
| FP16 | Yes |
| Gradient clipping | 1.0 |
| Warmup steps | 500 |
| Early stopping | on eval_loss (patience 3) |

In [ ]:
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    """
    Data collator that pads input_features and labels for Whisper.
    """
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        
        input_features = [{"input_features": f["input_features"]} for f in features]
        label_features = [{"input_ids":      f["labels"]}         for f in features]

        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )

        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch


data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)
print("Data collator ready.")

In [ ]:
def compute_metrics(pred):
    """Compute WER during training."""
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    label_ids[label_ids == -100] = tokenizer.pad_token_id

    pred_str  = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    pred_str  = [normalize_text(p) for p in pred_str]
    label_str = [normalize_text(l) for l in label_str]

    wer = 100 * wer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

In [ ]:

model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)

model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(
    language=LANGUAGE, task=TASK
)
model.config.suppress_tokens = []
model.config.use_cache = False   

print(f"Model loaded: {MODEL_NAME}")

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir=str(OUTPUT_DIR),
    per_device_train_batch_size=TRAIN_BATCH,
    per_device_eval_batch_size=EVAL_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    max_grad_norm=MAX_GRAD_NORM,
    num_train_epochs=NUM_EPOCHS,
    fp16=FP16,
    evaluation_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=3,
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    predict_with_generate=True,
    generation_max_length=225,
    report_to=["tensorboard"],
    push_to_hub=False,
    remove_unused_columns=False,
    dataloader_num_workers=2,
)

print("Training arguments configured.")

In [ ]:
trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor.feature_extractor,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

print("Starting fine-tuning...")
train_result = trainer.train()
print(f"Training complete.  Final metrics: {train_result.metrics}")

---
## 10 - Post-Training Evaluation and WER Report

Evaluate the **fine-tuned** model on your dataset's validation split and compare with the baseline.


In [ ]:

model_ft = model.to(device)
finetuned_wer = evaluate_model(model_ft, processor, eval_dataset)
print(f"Fine-tuned WER on your dataset: {finetuned_wer}%")


In [ ]:

def print_wer_report(baseline: float, finetuned: float):
    improvement = baseline - finetuned
    rel_improve = (improvement / baseline) * 100 if baseline > 0 else 0

    print("\n" + "=" * 60)
    print("             WER COMPARISON REPORT")
    print("=" * 60)
    print(f"{'Model':<35} {'Dataset':<18} {'WER (%)':<10}")
    print("-" * 60)
    print(f"{'Whisper-small (Pretrained)':<35} {'Your Dataset':<18} {baseline:<10}")
    print(f"{'Whisper-small (Fine-tuned)':<35} {'Your Dataset':<18} {finetuned:<10}")
    print("-" * 60)
    print(f"{'Absolute improvement':<35} {'':<18} {improvement:+.2f}")
    print(f"{'Relative improvement':<35} {'':<18} {rel_improve:.1f}%")
    print("=" * 60)


print_wer_report(baseline_wer, finetuned_wer)


---
## 11 - Save / Push Model

In [ ]:

model.save_pretrained(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
print(f"Model and processor saved to {OUTPUT_DIR}")

In [ ]:
print("Uncomment the block above to push to HuggingFace Hub.")

---

## Summary

| # | Step | Status |
|---|------|--------|
| 1 | Setup and Installation | Auto |
| 2 | Configuration | Set your Excel file + column names |
| 3 | Load Dataset from Excel | Auto |
| 3.1 | Fix Faulty Dataset URLs | Auto |
| 4 | Audio Preprocessing | Auto |
| 5 | Text Normalization | Auto |
| 6 | Quality Checks | Auto |
| 7 | Dataset Formatting | Auto |
| 8 | Baseline Eval | Auto (Your Excel Dataset) |
| 9 | Fine-Tuning | Auto |
| 10 | Eval and WER Report | Auto |
| 11 | Save / Push | Auto (save local) |

> **Usage**: Set your Excel filename and column names in Section 2,
> then run all cells top-to-bottom.
